# Load segmented volume into a Phantom Object


in this notebook , we will show case how can one load a segmented volume into the simulation software and create a phantom Object using our API.

To Show case this , first we will create a phantom, then createa segmented volume based on it and then load this segmented volume into the software

In [1]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from neutron_xray_sim.phantom import make_composite_phantom
from neutron_xray_sim.materials import MATERIALS

from neutron_xray_sim.volume_importer import (
    phantom_from_array,
    phantom_from_segmented_volume,
)

# Create a Phantom

In [2]:
phantom_original = make_composite_phantom(N=64)

phantom_original

PhantomData('composite', shape=(Nz=64, Nx=64, Ny=64), 0.015625 cm/voxel, materials=[Air, Al, HDPE, H₂O, Fe, Ti])

In [3]:
for i, mat in enumerate(phantom_original.materials):
    print(i, mat.name, mat.symbol, mat.mu_n)

0 Air Air 0.0
1 Aluminum Al 0.098
2 HDPE HDPE 2.18
3 Water H₂O 1.38
4 Iron Fe 1.16
5 Titanium Ti 0.64


In [4]:
seg_from_labels = phantom_original.label_vol.copy()

print(seg_from_labels.shape)
print(seg_from_labels.dtype)
print("Unique labels:", np.unique(seg_from_labels))

(64, 64, 64)
uint8
Unique labels: [0 1 2 3 4 5]


In [5]:
def material_to_database_key(material):
    """
    Find the key in MATERIALS corresponding to a Material object.

    This is safer than assuming material.name is exactly the same as the
    MATERIALS dictionary key.
    """
    for key, db_material in MATERIALS.items():
        if db_material is material:
            return key

    # fallback: try name matching
    name_key = material.name.lower()
    if name_key in MATERIALS:
        return name_key

    raise ValueError(f"Could not find material in MATERIALS database: {material}")


class_map = {
    int(label): material_to_database_key(phantom_original.materials[int(label)])
    for label in np.unique(seg_from_labels)
}

metadata_dict = {
    "name": "reimported_composite_from_labels",
    "voxel_cm": phantom_original.voxel_cm,
    "axis_order": "zxy",
    "class_map": class_map,
}

metadata_dict

{'name': 'reimported_composite_from_labels',
 'voxel_cm': 0.015625,
 'axis_order': 'zxy',
 'class_map': {0: 'air',
  1: 'aluminum',
  2: 'hdpe',
  3: 'water',
  4: 'iron',
  5: 'titanium'}}

# Import the created segmented volume

In [6]:
phantom_imported = phantom_from_array(
    volume=seg_from_labels,
    metadata=metadata_dict,
)

phantom_imported

PhantomData('reimported_composite_from_labels', shape=(Nz=64, Nx=64, Ny=64), 0.015625 cm/voxel, materials=[Air, Al, HDPE, H₂O, Fe, Ti])

# Compare original phantom and the imported 

In [7]:
print("Original shape:", phantom_original.shape)
print("Imported shape:", phantom_imported.shape)

print("Original voxel_cm:", phantom_original.voxel_cm)
print("Imported voxel_cm:", phantom_imported.voxel_cm)

print("Original materials:")
for i, mat in enumerate(phantom_original.materials):
    print(i, mat.name)

print("\nImported materials:")
for i, mat in enumerate(phantom_imported.materials):
    print(i, mat.name)

Original shape: (64, 64, 64)
Imported shape: (64, 64, 64)
Original voxel_cm: 0.015625
Imported voxel_cm: 0.015625
Original materials:
0 Air
1 Aluminum
2 HDPE
3 Water
4 Iron
5 Titanium

Imported materials:
0 Air
1 Aluminum
2 HDPE
3 Water
4 Iron
5 Titanium
